# LLDP demo 能力

## LLDP 是什么
LLDP（Link Layer Discovery Protocol） 是一种网络层协议

主要用于在同一局域网内的网络设备之间发现彼此的身份和拓扑结构信息。
LLDP 是 IEEE 802.1AB 标准的一部分，它允许网络设备（如交换机、路由器、无线接入点、服务器等）在数据链路层交换信息，以便彼此了解网络拓扑、设备信息、接口详情等

因此可以基于LLDP协议获取网络的物理拓扑关系，获取集群的交换机、服务器、rank关系


## 环境搭建
使用scapy包，基于此获取集群信息

In [ ]:
pip install scapy

## 捕获LLDP包
通过scapy，启动一个监听的进程，捕获LLDP包
注意，一台服务器上有多个可被监听的接口。
但是有些是localhost、docker等，这些接口无法捕获到LLDP包
需要找到对外的接口

In [ ]:
from scapy.all import sniff, Ether, raw

# 回调函数用于处理捕获的数据包
def lldp_packet_handler(packet):
    # 检查数据包是否是以太网数据包并且包含 LLDP 类型
    if packet.haslayer(Ether) and packet.type == 0x88CC:  # 0x88CC 是 LLDP 的以太网类型
        print("Captured LLDP packet:")
        packet.show()

# 捕获网络接口上的 LLDP 数据包
# sniff(iface="你的网络接口", filter="ether proto 0x88CC", prn=lldp_packet_handler, store=0)

sniff(iface="enp189s0f0", filter="ether proto 0x88CC", prn=lldp_packet_handler, store=0)

如果出现 ValueError: Interface 'xxxx' not found !
或者进程启动一分钟后无捕获到的数据（LLDP通常30s/60s会发送一次）

说明接口选择错误
（也有可能是设备未启动LLDP，假设集群环境的设备已启动LLDP）

可以使用以下方式查看本地有哪些接口
优先尝试enp开头的接口

In [ ]:
ip link show

In [ ]:
from scapy.all import get_if_list
print(get_if_list())

## 解析 LLDP
从LLDP包获取 IP、MAC地址等信息
其中解析出的数据量较多，需要评估哪些用于实际的拓扑

协议可以解析出的内容
Chassis ID：设备的唯一标识符（通常是 MAC 地址）。
Port ID：设备的端口标识符。
System Name：设备的名称（主机名）。
...

拓扑需要的信息
节点：每个设备的唯一标识符（通常是 Chassis ID 或 System Name）。
边：设备端口（Port ID）之间的连接，表示设备间的物理连接关系

In [ ]:
pip install networkx matplotlib

In [ ]:
import scapy.all as scapy
import networkx as nx
import matplotlib.pyplot as plt
from scapy import Ether
from scapy.contrib.lldp import LLDPDU

# 存储网络拓扑的字典：{设备的唯一标识符: {端口: 连接的设备标识符}}
topology = {}

# 解析 LLDP 数据包
def parse_lldp_packet(packet):
    """
    解析 LLDP 包，提取设备信息并更新网络拓扑。
    """
    if packet.haslayer(Ether):
        # 获取源 MAC 和目标 MAC 地址
        src_mac = packet[Ether].src
        dst_mac = packet[Ether].dst

    if packet.haslayer(LLDPDU):
        # 获取 LLDP 的各个 TLV 数据字段
        lldp = packet[LLDPDU]

        # 设备的 Chassis ID (通常是源设备的 MAC 地址)
        chassis_id = lldp.chassis_id

        # 端口 ID (源设备的端口)
        port_id = lldp.port_id

        # 系统名称 (设备的主机名)
        system_name = lldp.system_name if hasattr(lldp, 'system_name') else "Unknown"

        # 更新网络拓扑
        if chassis_id not in topology:
            topology[chassis_id] = {}

        topology[chassis_id][port_id] = system_name  # 设备通过端口与其他设备连接

        # 输出当前解析的设备信息
        print(f"Device {system_name} (Chassis ID: {chassis_id}) on Port {port_id}")

# 捕获 LLDP 数据包并解析
def capture_lldp_packets(interface="eth0", packet_count=100):
    """
    捕获指定接口上的 LLDP 数据包并解析。
    """
    scapy.sniff(iface="ens835f0", filter="ether proto 0x88cc", count=packet_count, prn=parse_lldp_packet, store=0)

# 绘制网络拓扑图
def draw_topology():
    """
    使用 networkx 和 matplotlib 绘制网络拓扑图。
    """
    G = nx.Graph()  # 创建无向图

    # 遍历拓扑字典，将每个设备及其连接加入图中
    for device, ports in topology.items():
        for port, connected_device in ports.items():
            G.add_edge(device, connected_device, label=f"Port {port}")

    # 绘制网络图
    pos = nx.spring_layout(G)  # 使用 spring 布局绘制节点
    nx.draw(G, pos, with_labels=True, node_size=3000, node_color="skyblue", font_size=10, font_weight="bold")
    labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)

    # 显示图形
    plt.title("Network Topology Based on LLDP")
    plt.show()

# 捕获 50 个 LLDP 数据包
capture_lldp_packets(interface="eth0", packet_count=50)

# 绘制网络拓扑图
draw_topology()